In [1]:
import pandas as pd
df = pd.read_parquet("../classify_data.parquet").reset_index()

In [2]:
class_map = {
    '0': 0,
    '1-3': 1,
    '4-7': 2,
    '8-14': 3,
    '15-30': 4,
    '31-60': 5
}
df['resolution_class_int'] = df['resolution_class'].map(class_map)

In [ ]:
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

feature_columns = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip',
                    'Incident Address', 'City', 'Borough',
                    'Open Data Channel Type', 'Latitude', 'Longitude',
                    'hour_of_day', 'day_of_week', 'month',
                    'is_weekend', 'open_requests_same_day_zip']

X = df[feature_columns]
y = df['resolution_class_int']

#Segregate columsn for preprocessing
categorical_cols = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip',
                    'Incident Address', 'City', 'Borough',
                    'Open Data Channel Type',
                    'is_weekend']
numeric_cols = ['Latitude','Longitude','hour_of_day','day_of_week', 'month','open_requests_same_day_zip']

# Preprocessing -> Normalization + encoding categories
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'  # numeric features untouched
)

xg = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",            
    enable_categorical=True,
    n_jobs=8,
    verbosity=3,
    random_state=42      
)

# 4. Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', xg)
])

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro')

print(scores)

# 6. Fit model
model.fit(X_train, y_train)

# 7. Predict
y_pred = model.predict(X_test)

[23:35:03] ======== Monitor (0): HostSketchContainer ========
[23:35:03] AllReduce: 0.003823s, 1 calls @ 3823us

[23:35:03] MakeCuts: 0.003854s, 1 calls @ 3854us

[23:35:03] INFO: /workspace/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (1272686, 16, 20362976).
[23:35:03] DEBUG: /workspace/src/gbm/gbtree.cc:127: Using tree method: 3
[23:35:48] ======== Monitor (0): GBTree ========
[23:35:48] BoostNewTrees: 41.0165s, 200 calls @ 41016518us

[23:35:48] CommitModel: 0.000295s, 200 calls @ 295us

[23:35:48] ======== Monitor (0): HistUpdater ========
[23:35:48] BuildHistogram: 13.0235s, 8400 calls @ 13023481us

[23:35:48] EvaluateSplits: 0.783138s, 9600 calls @ 783138us

[23:35:48] InitData: 0.853107s, 1200 calls @ 853107us

[23:35:48] InitRoot: 3.58249s, 1200 calls @ 3582486us

[23:35:48] LeafPartition: 0.000509s, 1200 calls @ 509us

[23:35:48] UpdatePosition: 13.5883s, 9600 calls @ 13588260us

[23:35:48] UpdatePredictionCache: 4.52885s, 1200 calls @ 45288

In [9]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# print(scores)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.8063479292122919

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.94      0.93    331345
           1       0.57      0.71      0.63     76042
           2       0.40      0.31      0.35     27143
           3       0.47      0.19      0.27     17086
           4       0.39      0.27      0.32     15160
           5       0.50      0.35      0.42     10482

    accuracy                           0.81    477258
   macro avg       0.54      0.46      0.49    477258
weighted avg       0.80      0.81      0.80    477258


Confusion Matrix:
 [[311663  17021   1309    202    448    702]
 [ 15672  53799   4084    630   1244    613]
 [  4092  11404   8306   1274   1493    574]
 [  2187   5863   3373   3260   1803    600]
 [  1914   4276   2571   1143   4098   1158]
 [  1350   2256   1287    398   1481   3710]]


In [10]:
import numpy as np
np.save("../results/xgboost_predictions_200.npy", y_pred)

In [8]:
import numpy as np
y_pred = np.load("../results/random_forest_predictions_150.npy")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt 

sns.heatmap(confusion_matrix(y_test, y_pred), annot=False)
plt.savefig("../results/cm_random_forest.png")

In [15]:
tes = df[categorical_cols]
tes.describe(include='all')

,Agency,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,City,Borough,Open Data Channel Type,is_weekend
count,2386288,2386288,2386288,2386288,2386288,2386288,2386288,2386288,2386288,2.386288e+06
unique,15,168,852,137,243,467120,51,6,5,NaN
top,NYPD,ILLEGAL PARKING,LOUD MUSIC/PARTY,STREET/SIDEWALK,10466,UNKNOWN,BROOKLYN,BROOKLYN,ONLINE,NaN
freq,1277321,435694,355810,870943,82021,77782,720072,720073,1041275,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.765500e-01
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.472921e-01
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00
